# Mantis Vision — Train in Google Colab

Run these cells top to bottom. Before starting: **Runtime -> Change runtime type -> GPU**.

This notebook: clones the repo, installs dependencies, pulls the labeled dataset down from Kaggle, trains the EfficientNet-B0 health classifier, evaluates it, and lets you download the resulting checkpoint.

## 1. Clone the repo

In [ ]:
!git clone https://github.com/Arran0/MantisVision.git
%cd MantisVision/ml

## 2. Install dependencies

Colab already has torch/torchvision preinstalled with GPU support, so this just adds the extra packages the pipeline needs.

In [ ]:
!pip install -q grad-cam kaggle fastapi python-multipart

## 3. Kaggle credentials

Get `kaggle.json` from kaggle.com/settings -> API -> "Create New Token" (downloads to your computer). Run this cell, then use the file picker to upload that `kaggle.json`.

In [ ]:
from google.colab import files
import os

uploaded = files.upload()  # pick kaggle.json when prompted
os.makedirs("/root/.kaggle", exist_ok=True)
!mv kaggle.json /root/.kaggle/kaggle.json
!chmod 600 /root/.kaggle/kaggle.json

## 4. Download the dataset from Kaggle

Replace `YOUR_USERNAME/YOUR_DATASET_SLUG` with the id from your Kaggle dataset's URL (`kaggle.com/datasets/<that part>`).

In [ ]:
DATASET_ID = "YOUR_USERNAME/YOUR_DATASET_SLUG"  # <-- edit this
!python scripts/kaggle_sync.py download --dataset {DATASET_ID}

## 5. Validate the dataset

Checks every class folder exists, every image opens correctly, and flags underrepresented classes.

In [ ]:
!python -m src.data.validate_dataset

## 6. Train

Two-phase schedule (frozen backbone, then fine-tune) with early stopping. Saves the best checkpoint to `ml/checkpoints/best_model.pt`. This can take a while depending on dataset size — Colab's free GPU tier helps a lot here.

In [ ]:
!python -m src.train

## 7. Evaluate

Accuracy/precision/recall/F1 (macro + per-class), confusion matrix, ROC AUC on the held-out test split.

In [ ]:
!python -m src.evaluate

from IPython.display import Image, display
display(Image(filename="reports/confusion_matrix.png"))

## 8. Download the trained checkpoint

Colab runtimes are ephemeral — grab the checkpoint now so you don't lose it when the session disconnects. You'll need this file to run the inference API later.

In [ ]:
from google.colab import files
files.download("checkpoints/best_model.pt")

## Optional: try a Grad-CAM explanation on one photo

Upload a single test photo and see the heatmap of what the model looked at.

In [ ]:
from google.colab import files
uploaded = files.upload()  # pick one photo
photo_path = list(uploaded.keys())[0]
!python -m src.gradcam "{photo_path}"

from IPython.display import Image, display
import pathlib
display(Image(filename=str(pathlib.Path(photo_path).with_suffix('.gradcam.png'))))